In [7]:
import sys
sys.path.append("/Users/evangelosdimitriou/myGitRepo/multi-task-GPs-for-Treatment-Effect-Estimation-2/__pycache__")

import numpy as np
import pandas as pd
import GPy
import copy
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge as ridge
from neurIPS.functions.data_simulation import data_simulation8
from neurIPS.functions.model_training import ICM

# -------------------------------
# Utility functions
# -------------------------------
rmse = lambda x, y: np.sqrt(mean_squared_error(x, y))

linear_reg_parameter_pairs = [
    (ridge(), {'alpha': np.logspace(-7, 7, 8), 'tol': [1e-3]})
]

def get_best_model_from_gridcv(X, Y, reg_parameter_pairs, print_flag=False, verbose=0):
    val_errs = []
    models = []
    x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2)
    for reg, parameters in reg_parameter_pairs:
        if print_flag:
            print('Trying', str(reg)[:50])
        model = GridSearchCV(reg, parameters, cv=2, refit=True, verbose=verbose, n_jobs=10)
        model.fit(X, Y)
        val_errs.append(rmse(y_test, model.predict(x_test)))
        models.append(copy.deepcopy(model.best_estimator_))
    min_ind = val_errs.index(min(val_errs))
    if verbose != 0:
        print(str(models[min_ind]), 'val rmse:{}'.format(val_errs[min_ind]))
    return copy.deepcopy(models[min_ind])

# -------------------------------
# GP-based Kallus adjustment
# -------------------------------
def run_methodGP(X_rct, Y_rct, T_rct, X_obs, Y_obs, T_obs, X_eval):
    # GP on observational data
    kern = GPy.kern.RBF(input_dim=1, variance=1., lengthscale=1.)
    f0_gp = GPy.models.GPRegression(np.vstack(X_obs[T_obs==0]), np.vstack(Y_obs[T_obs==0]), kern)
    f0_gp.optimize()
    f1_gp = GPy.models.GPRegression(np.vstack(X_obs[T_obs==1]), np.vstack(Y_obs[T_obs==1]), kern)
    f1_gp.optimize()

    # Predict CATE components
    omega_rct_pred = f1_gp.predict(np.vstack(X_rct))[0] - f0_gp.predict(np.vstack(X_rct))[0]
    omega_obs_pred = f1_gp.predict(np.vstack(X_obs))[0] - f0_gp.predict(np.vstack(X_obs))[0]
    omega_eval_pred = f1_gp.predict(np.vstack(X_eval))[0] - f0_gp.predict(np.vstack(X_eval))[0]

    # Estimate residual eta on RCT
    prop_rct = np.mean(T_rct)
    def ite_adjusted_outcome(Y, T, _propensity, c=None):
        c = np.mean(Y) if c is None else c
        _ite = np.zeros(Y.shape)
        _ite[T==1] = (Y[T==1] - c) / _propensity
        _ite[T==0] = -(Y[T==0] - c) / (1-_propensity)
        return _ite

    cate_rct_est = ite_adjusted_outcome(Y_rct, T_rct, _propensity=prop_rct)
    eta_rct_est = np.vstack(cate_rct_est) - np.vstack(omega_rct_pred)

    # Fit linear model to eta
    best_eta_model = get_best_model_from_gridcv(X_rct.reshape(-1,1), eta_rct_est, linear_reg_parameter_pairs)
    
    return copy.deepcopy(best_eta_model), omega_obs_pred, omega_eval_pred

# -------------------------------
# Initialize arrays to store results
# -------------------------------
num_simulations = 100
test_data = np.round(np.arange(-2.0, 2.04, 0.04), 2).reshape(-1,1)

best_rho = pd.read_csv('sim8_best_rho.csv').values[0][0]

df_all = pd.read_csv("/Users/evangelosdimitriou/myGitRepo/CausalICM/univariate/Simulation 8/datasets_sim8.csv")

    



In [11]:
import time

runtime_results = {
    "datasetNo": [],
    "naiveGP_exp": [],
    "naiveGP_obs": [],
    "kallusGP": [],
    "ICM": []
}


In [12]:
num_simulations = 100
test_data = np.round(np.arange(-2.0, 2.04, 0.04), 2).reshape(-1,1)

for i in range(num_simulations):
    print(f"Simulation {i+1}")

    data_exp = df_all[(df_all["S"]==1) & (df_all["datasetNo"]==i+1)]
    data_obs = df_all[(df_all["S"]==0) & (df_all["datasetNo"]==i+1)]

    X0_obs, X1_obs = data_obs[data_obs["A"]==0]["X"], data_obs[data_obs["A"]==1]["X"]
    Y0_obs, Y1_obs = data_obs[data_obs["A"]==0]["Y"], data_obs[data_obs["A"]==1]["Y"]
    X0_exp, X1_exp = data_exp[data_exp["A"]==0]["X"], data_exp[data_exp["A"]==1]["X"]
    Y0_exp, Y1_exp = data_exp[data_exp["A"]==0]["Y"], data_exp[data_exp["A"]==1]["Y"]

    trueCATE = 1 + test_data + test_data**2

    runtime_results["datasetNo"].append(i+1)

    # ===============================
    # 1) Naive GP (observational)
    # ===============================
    start = time.perf_counter()

    kern = GPy.kern.RBF(input_dim=1, variance=1., lengthscale=1.)
    m0_obs = GPy.models.GPRegression(np.vstack(X0_obs), np.vstack(Y0_obs), kern)
    m0_obs.optimize()
    m1_obs = GPy.models.GPRegression(np.vstack(X1_obs), np.vstack(Y1_obs), kern)
    m1_obs.optimize()
    naiveCATE_obs = (m1_obs.predict(np.vstack(test_data))[0] - 
                     m0_obs.predict(np.vstack(test_data))[0]).flatten()

    runtime_results["naiveGP_obs"].append(time.perf_counter() - start)

    # ===============================
    # 2) Naive GP (experimental)
    # ===============================
    start = time.perf_counter()

    m0_exp = GPy.models.GPRegression(np.vstack(X0_exp), np.vstack(Y0_exp), kern)
    m0_exp.optimize()
    m1_exp = GPy.models.GPRegression(np.vstack(X1_exp), np.vstack(Y1_exp), kern)
    m1_exp.optimize()
    naiveCATE_exp = (m1_exp.predict(np.vstack(test_data))[0] - 
                     m0_exp.predict(np.vstack(test_data))[0]).flatten()

    runtime_results["naiveGP_exp"].append(time.perf_counter() - start)

    # ===============================
    # 3) Kallus GP
    # ===============================
    start = time.perf_counter()

    eta_model, omega_obs_pred, omega_eval_pred = run_methodGP(
        data_exp["X"].values, data_exp["Y"].values, data_exp["A"].values,
        data_obs["X"].values, data_obs["Y"].values, data_obs["A"].values,
        test_data
    )
    tau_pred = eta_model.predict(test_data.reshape(-1,1)) + omega_eval_pred

    runtime_results["kallusGP"].append(time.perf_counter() - start)

    # ===============================
    # 4) ICM
    # ===============================
    start = time.perf_counter()

    m0_icm, m1_icm = ICM(X_E=data_exp["X"], X_O=data_obs["X"],
                        T_E=data_exp["A"], T_O=data_obs["A"],
                        Y_E=data_exp["Y"], Y_O=data_obs["Y"],
                        r=2, ID=1, AD=0, rho=best_rho)

    predCATE_icm = m1_icm.predict_noiseless(np.c_[np.vstack(test_data), np.zeros(test_data.shape[0])])[0] - \
                   m0_icm.predict_noiseless(np.c_[np.vstack(test_data), np.zeros(test_data.shape[0])])[0]

    runtime_results["ICM"].append(time.perf_counter() - start)


Simulation 1
Simulation 2
Simulation 3
Simulation 4
Simulation 5


 /Users/evangelosdimitriou/anaconda3/lib/python3.10/site-packages/paramz/transformations.py:111: RuntimeWarning:overflow encountered in expm1


Simulation 6
Simulation 7


 /Users/evangelosdimitriou/anaconda3/lib/python3.10/site-packages/paramz/transformations.py:111: RuntimeWarning:overflow encountered in expm1


Simulation 8
Simulation 9
Simulation 10
Simulation 11
Simulation 12


...

Simulation 88
Simulation 89
Simulation 90
Simulation 91
Simulation 92


 /Users/evangelosdimitriou/anaconda3/lib/python3.10/site-packages/paramz/transformations.py:111: RuntimeWarning:overflow encountered in expm1


Simulation 93
Simulation 94


 /Users/evangelosdimitriou/anaconda3/lib/python3.10/site-packages/paramz/transformations.py:111: RuntimeWarning:overflow encountered in expm1


Simulation 95
Simulation 96
Simulation 97
Simulation 98
Simulation 99
Simulation 100


In [13]:
runtime_df = pd.DataFrame(runtime_results)
runtime_df.head()


,datasetNo,naiveGP_exp,naiveGP_obs,kallusGP,ICM
0,1,0.237389,1.736373,10.213725,5.433369
1,2,0.305404,1.717664,1.806594,3.730143
2,3,0.394719,2.233455,2.610087,3.765791
3,4,0.235527,2.313867,2.417199,3.723632
4,5,0.106824,2.597414,2.307151,4.940378


In [14]:
runtime_summary = runtime_df.drop(columns=["datasetNo"]).agg(["mean", "std", "median", "min", "max"]).T

runtime_summary = runtime_summary.rename(columns={
    "mean": "Mean (s)",
    "std": "SD (s)",
    "median": "Median (s)",
    "min": "Min (s)",
    "max": "Max (s)"
})
runtime_summary

,Mean (s),SD (s),Median (s),Min (s),Max (s)
naiveGP_exp,0.244369,0.114056,0.236458,0.089087,0.489595
naiveGP_obs,2.232287,0.639261,2.077287,1.417782,6.207515
kallusGP,2.394518,0.951331,2.181754,1.566869,10.213725
ICM,4.559152,0.782487,4.513439,3.310538,8.185129


In [15]:
runtime_df.to_csv("runtime_comparison_results.csv", index=False)


In [1]:
import pandas as pd
runtime_df = pd.read_csv("runtime_comparison_results.csv")
runtime_df.head()

,datasetNo,naiveGP_exp,naiveGP_obs,kallusGP,ICM
0,1,0.237389,1.736373,10.213725,5.433369
1,2,0.305404,1.717664,1.806594,3.730143
2,3,0.394719,2.233455,2.610087,3.765791
3,4,0.235527,2.313867,2.417199,3.723632
4,5,0.106824,2.597414,2.307151,4.940378
